In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [3]:
file = "/content/drive/MyDrive/dataset/network_anomaly_knn_dataset.csv"

In [4]:
df = pd.read_csv(file)

In [6]:
print(df.shape)

(100000, 17)


In [7]:
df.head(100)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,count,srv_count,same_srv_rate,diff_srv_rate,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,dst_host_count,dst_host_srv_count,label
0,0.938536,1,9,1,6256.066229,84.804039,32,33,0.213175,0.499363,0.564625,0.487643,0.799460,0.579695,175,131,0
1,6.020243,1,8,2,11448.941220,3413.819535,82,23,0.056268,0.031824,0.041356,0.078923,0.413426,0.517496,243,247,1
2,2.633491,2,7,1,2251.091399,406.396390,66,51,0.667049,0.157650,0.355760,0.646926,0.423866,0.097938,84,146,0
3,1.825885,0,2,0,751.217259,7344.757738,72,58,0.716493,0.130214,0.221439,0.022260,0.145328,0.770492,54,73,0
4,0.339250,0,4,5,3862.430503,271.636269,23,81,0.730571,0.123889,0.661467,0.859172,0.772688,0.944662,36,177,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1.361629,1,0,0,2310.388913,1946.423835,40,61,0.408937,0.722604,0.187285,0.324684,0.679442,0.183490,149,139,0
96,1.479358,2,4,5,1229.292445,8661.584099,48,86,0.543874,0.878955,0.383135,0.779460,0.875332,0.813454,114,86,0
97,1.115628,1,7,0,707.437422,1812.814992,54,34,0.692020,0.786870,0.575697,0.961552,0.795169,0.773410,136,228,0
98,0.051496,0,1,2,1279.199890,549.231305,96,39,0.352275,0.417424,0.732984,0.872252,0.123031,0.294149,49,146,1


In [11]:
X = df.drop("label", axis=1)
y = df["label"]

In [17]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
scores = []

for k in range(1, 21):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)

    scores.append(acc)

print("Accuracy for different K values:")
print(scores)

best_k = scores.index(max(scores)) + 1

print("\nBest K:", best_k)


Accuracy for different K values:
[0.84, 0.84185, 0.86575, 0.85425, 0.8707, 0.861, 0.87515, 0.8641, 0.87625, 0.865, 0.8748, 0.8661, 0.8767, 0.86615, 0.8741, 0.86705, 0.8743, 0.86675, 0.8747, 0.8657]

Best K: 13


In [24]:
model = KNeighborsClassifier(
    n_neighbors=best_k,
    algorithm="kd_tree",
    weights="distance",
    n_jobs=-1
)

model.fit(X_train, y_train)

KNeighborsClassifier(algorithm='kd_tree', n_jobs=-1, n_neighbors=13,
                     weights='distance')

In [28]:
pred = model.predict(X_test)

print("\nFinal Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred))


Final Accuracy: 0.877

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.98      0.93     15733
           1       0.89      0.48      0.63      4267

    accuracy                           0.88     20000
   macro avg       0.88      0.73      0.78     20000
weighted avg       0.88      0.88      0.86     20000



In [31]:
joblib.dump(model, "knn_ids_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel saved as knn_ids_model.pkl")
print("Scaler saved as scaler.pkl")


Model saved as knn_ids_model.pkl
Scaler saved as scaler.pkl
